# 02 - Data Cleaning

This notebook loads the 100,000-row sample from notebook 01 and applies a
comprehensive cleaning pipeline: column renaming, null imputation, outlier capping,
data type fixes, binary encoding, and rounding.

`student_id` is retained throughout — it was already reassigned to a clean
sequential range (1..N) in notebook 01.

## 1. Imports and Load Sampled Data

In [65]:
import os
import pathlib
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# Restore the sampled dataframe from notebook 01
%store -r df_sample
print(f"df_sample loaded: {df_sample.shape}")
df_sample.head(2)

df_sample loaded: (100000, 48)


,student_id,country,development_level,poverty_rate_percent,internet_infrastructure_index,average_internet_speed_mbps,age,gender,urban_rural,family_income_level,device_access,internet_access_hours,education_level,field_of_study,academic_motivation,online_learning_hours,social_media_hours,sessions_per_day,average_session_length_minutes,late_night_usage,education_content_hours,short_video_hours,entertainment_content_hours,news_content_hours,likes_given_per_day,comments_written_per_day,posts_created_per_week,late_night_score,brain_rot_index,brain_rot_level,attention_span_minutes,study_hours_per_week,class_attendance_rate,productivity_score,sleep_hours,stress_level,anxiety_score,depression_score,ads_viewed_per_day,ads_clicked_per_week,impulse_purchase_score,digital_spending_per_month,cyberbullying_exposure,adult_content_exposure,digital_addiction_score,wellbeing_index,academic_risk_score,financial_risk_score
0,1,Canada,Developed,14.51,90.50,188.19,15,Female,Urban,High,Both,6.249468,School,NaN,6.390075,6.704441,4.746323,13.971797,56.627988,Sometimes,0.942072,2.800652,1.003598,0.548203,132.141226,16.082200,4.338719,1,25.065907,Low,56.949323,26.245856,90.681705,8.453104,5.850479,5.328371,6.040984,8.434217,139.742376,13.717203,5.393212,106.593080,No,No,17.143210,54.673310,0.0,36.330249
1,2,Vietnam,Developing,30.70,66.87,45.63,21,Female,Urban,Middle,Smartphone,6.902865,Graduate,Social Sciences,8.003923,7.355535,4.274599,11.720024,53.355299,Often,0.917838,1.419909,1.936852,0.504641,94.866403,4.059623,7.077895,2,25.947886,Low,56.289038,24.867465,98.829238,8.196875,5.544846,5.684441,5.046559,7.242233,122.072394,14.108032,7.362823,82.212382,No,No,19.323299,49.714048,0.0,53.896830


## 2. Column Cleanup

### 2.1 Rename Columns
All column names are converted to lowercase with words separated by underscores,
matching the snake_case convention used throughout this pipeline.

In [66]:
df = df_sample.copy()

# Normalize: strip whitespace, lowercase, replace spaces/special chars with underscore
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

print("Renamed columns:")
print(list(df.columns))

Renamed columns:
['student_id', 'country', 'development_level', 'poverty_rate_percent', 'internet_infrastructure_index', 'average_internet_speed_mbps', 'age', 'gender', 'urban_rural', 'family_income_level', 'device_access', 'internet_access_hours', 'education_level', 'field_of_study', 'academic_motivation', 'online_learning_hours', 'social_media_hours', 'sessions_per_day', 'average_session_length_minutes', 'late_night_usage', 'education_content_hours', 'short_video_hours', 'entertainment_content_hours', 'news_content_hours', 'likes_given_per_day', 'comments_written_per_day', 'posts_created_per_week', 'late_night_score', 'brain_rot_index', 'brain_rot_level', 'attention_span_minutes', 'study_hours_per_week', 'class_attendance_rate', 'productivity_score', 'sleep_hours', 'stress_level', 'anxiety_score', 'depression_score', 'ads_viewed_per_day', 'ads_clicked_per_week', 'impulse_purchase_score', 'digital_spending_per_month', 'cyberbullying_exposure', 'adult_content_exposure', 'digital_addict

### 2.2 Drop Irrelevant Columns

`student_id` is **kept** — it was reassigned as a clean sequential ID in notebook 01
and serves as the row identifier throughout the pipeline.

Columns are only dropped if they exceed 70% missing values, as they offer
insufficient signal for analysis or modelling.

In [67]:
# Drop only columns that exceed the 70% null threshold
high_null_cols = df.columns[df.isnull().mean() > 0.70].tolist()

if high_null_cols:
    print(f"Columns with >70% nulls (will be dropped): {high_null_cols}")
    df.drop(columns=high_null_cols, inplace=True)
else:
    print("No columns exceed the 70% null threshold. Nothing dropped.")

print(f"Shape after column drop : {df.shape}")
print(f"student_id retained     : {'student_id' in df.columns}")

No columns exceed the 70% null threshold. Nothing dropped.
Shape after column drop : (100000, 48)
student_id retained     : True


## 3. Null Value Treatment

### 3.1 Standardize Null Sentinels

Many datasets encode missing values as strings like 'NA', 'N/A', '-', etc.
All such values are first converted to proper `NaN` before imputation.

`student_id` is excluded from sentinel replacement since it is a numeric ID column.

In [68]:
# Record null counts before treatment
null_before = df.isnull().sum()

# Sentinel strings that represent missing data
NULL_SENTINELS = ["", "NA", "N/A", "na", "n/a", "not applicable",
                  "Not Applicable", "NULL", "null", "-", "."]

# Replace sentinels with NaN in object (string) columns only
# student_id is numeric so it will not be affected
obj_cols = df.select_dtypes(include="object").columns
df[obj_cols] = df[obj_cols].replace(NULL_SENTINELS, np.nan)

# Also strip leading/trailing whitespace from string columns
for col in obj_cols:
    df[col] = df[col].str.strip() if df[col].dtype == object else df[col]

null_after_sentinel = df.isnull().sum()
additional = (null_after_sentinel - null_before).sum()
print(f"Additional NaNs introduced by sentinel replacement: {additional}")

Additional NaNs introduced by sentinel replacement: 0


### 3.2 Impute Missing Values

- Numeric columns: filled with the **median** (robust to skew and outliers)
- Categorical columns: filled with the **mode** (most frequent category)
- `student_id` is excluded from imputation as it should have no nulls
- `brain_rot_level` is excluded here — it will be re-derived from `brain_rot_index` in Section 5.4

In [69]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include="object").columns.tolist()

# Impute numeric columns with median (skip student_id — it is an identifier, not a measure)
for col in num_cols:
    if col == "student_id":
        continue
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)

# Impute categorical columns with mode
# brain_rot_level is intentionally skipped — it will be rederived from brain_rot_index
for col in cat_cols:
    if col == "brain_rot_level":
        continue
    if df[col].isnull().any():
        mode_val = df[col].mode()
        if not mode_val.empty:
            df[col].fillna(mode_val[0], inplace=True)

null_after_impute = df.isnull().sum()

# Before / after comparison table
comparison = pd.DataFrame({
    "null_before": null_before,
    "null_after":  null_after_impute
})
comparison = comparison[comparison["null_before"] > 0]

if comparison.empty:
    print("No null values existed before imputation.")
else:
    print("Before / after null counts:")
    print(comparison)

remaining_nulls = df.isnull().sum().sum()
print(f"\nTotal remaining nulls after imputation: {remaining_nulls}")
print("(brain_rot_level nulls will be resolved in Section 5.4)")

Before / after null counts:
                 null_before  null_after
field_of_study         49584           0
brain_rot_level         1235        1235

Total remaining nulls after imputation: 1235
(brain_rot_level nulls will be resolved in Section 5.4)


## 4. Outlier Detection and Treatment

The IQR (Interquartile Range) method is used to identify outliers in each numeric column.
Outliers are **capped** (Winsorized) at the 1.5 x IQR bounds — no rows are deleted.

`student_id` is excluded from outlier treatment since it is an identifier, not a measure.

In [70]:
def iqr_bounds(series: pd.Series, factor: float = 1.5):
    """Return (lower, upper) Winsorization bounds using the IQR method."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - factor * iqr, q3 + factor * iqr


outlier_report = []

# Exclude student_id from outlier treatment — it is a sequential identifier
measure_cols = [c for c in num_cols if c != "student_id"]

for col in measure_cols:
    lower, upper = iqr_bounds(df[col])
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report.append({"column": col, "outliers": n_outliers,
                            "lower_bound": round(lower, 4), "upper_bound": round(upper, 4)})
    # Cap values to bounds
    df[col] = df[col].clip(lower=lower, upper=upper)

outlier_df = pd.DataFrame(outlier_report).set_index("column")
print("Outlier counts and Winsorization bounds per numeric column:")
print(outlier_df[outlier_df["outliers"] > 0].sort_values("outliers", ascending=False))

total_outliers = outlier_df["outliers"].sum()
print(f"\nTotal outlier values capped: {total_outliers:,}")

Outlier counts and Winsorization bounds per numeric column:
                                outliers  lower_bound  upper_bound
column                                                            
education_content_hours             3446      -0.3230       1.3327
academic_risk_score                 3421       0.0000       0.0000
average_internet_speed_mbps         1994    -110.0650     237.8950
digital_spending_per_month          1954     -19.5842     134.5924
posts_created_per_week              1608      -2.0627      10.0931
short_video_hours                   1544      -0.2599       3.2076
news_content_hours                  1461      -0.0841       1.1036
likes_given_per_day                 1422     -12.3710     147.2913
entertainment_content_hours         1415      -0.5704       3.0872
comments_written_per_day            1118      -4.1965      17.3898
productivity_score                   868       5.0401      12.9759
financial_risk_score                 793       4.7318      77.9362
ad

## 5. Data Type Fixes


### 5.1 Ensure Numeric Columns are Numeric
Object-typed columns that are actually numeric (e.g., stored as strings) are coerced.

In [71]:
# Re-check object columns after sentinel replacement — some may be numeric
# Exclude brain_rot_level — it is a categorical label, not a numeric column
obj_cols_now = [c for c in df.select_dtypes(include="object").columns
                if c != "brain_rot_level"]

for col in obj_cols_now:
    converted = pd.to_numeric(df[col], errors="coerce")
    # Only convert if the result is mostly non-null (>90% valid numeric)
    if converted.notna().mean() > 0.90:
        df[col] = converted
        print(f"Converted '{col}' to numeric.")

print("Numeric coercion step complete.")

Numeric coercion step complete.


### 5.2 Encode Binary Categoricals
Columns containing only Yes/No (or similar binary string values) are mapped to 1/0.

In [72]:
BINARY_MAP = {
    "yes": 1, "no": 0,
    "true": 1, "false": 0,
    "y": 1, "n": 0
}

obj_cols_now = df.select_dtypes(include="object").columns
encoded_cols = []

for col in obj_cols_now:
    unique_vals = df[col].dropna().str.lower().unique()
    # Column qualifies if all unique values map to 0 or 1
    if set(unique_vals).issubset(BINARY_MAP.keys()):
        df[col] = df[col].str.lower().map(BINARY_MAP)
        encoded_cols.append(col)

if encoded_cols:
    print(f"Binary-encoded columns (Yes/No -> 1/0): {encoded_cols}")
else:
    print("No binary-only categorical columns found.")

Binary-encoded columns (Yes/No -> 1/0): ['cyberbullying_exposure', 'adult_content_exposure']


### 5.3 Derive brain_rot_level from brain_rot_index

The `brain_rot_level` column is recalculated from `brain_rot_index` using the
following business rules, overwriting any existing or null values:

| brain_rot_index | brain_rot_level |
|---|---|
| <= 15 | Low |
| 15 < index <= 30 | Medium |
| > 30 | High |

In [73]:
def classify_brain_rot(index_value):
    """Map a brain_rot_index value to its corresponding level label."""
    if pd.isna(index_value):
        return np.nan
    if index_value <= 15:
        return "Low"
    elif index_value <= 30:
        return "Medium"
    else:
        return "High"


# Rederive brain_rot_level — this also resolves any nulls that were skipped in imputation
df["brain_rot_level"] = df["brain_rot_index"].apply(classify_brain_rot)

# Verify the distribution matches expectations
level_counts = df["brain_rot_level"].value_counts()
print("brain_rot_level distribution after rederivation:")
print(level_counts)
print(f"\nNull count in brain_rot_level: {df['brain_rot_level'].isnull().sum()}")

# Sanity check: confirm thresholds are correct
low_max    = df.loc[df["brain_rot_level"] == "Low",    "brain_rot_index"].max()
medium_max = df.loc[df["brain_rot_level"] == "Medium", "brain_rot_index"].max()
high_min   = df.loc[df["brain_rot_level"] == "High",   "brain_rot_index"].min()
print(f"\nSanity check:")
print(f"  Max brain_rot_index in Low    : {low_max:.4f}  (expected <= 15)")
print(f"  Max brain_rot_index in Medium : {medium_max:.4f}  (expected <= 30)")
print(f"  Min brain_rot_index in High   : {high_min:.4f}  (expected  > 30)")

brain_rot_level distribution after rederivation:
brain_rot_level
Medium    53055
Low       34592
High      12353
Name: count, dtype: int64

Null count in brain_rot_level: 0

Sanity check:
  Max brain_rot_index in Low    : 15.0000  (expected <= 15)
  Max brain_rot_index in Medium : 29.9997  (expected <= 30)
  Min brain_rot_index in High   : 30.0000  (expected  > 30)


### 5.4 Engineer academic_risk_score

The `academic_risk_score` column originally contains only 0.0 values.
It is recalculated using a relevant weighted formula based on other columns:
`(100 - class_attendance_rate) * 0.4 + (10 - academic_motivation) * 2.5 + (digital_addiction_score) * 0.5 + (brain_rot_index) * 0.3`
The score is then clipped at 0 to prevent negative values.

In [74]:
df["academic_risk_score"] = (
    (100 - df["class_attendance_rate"]) * 0.4 +
    (10 - df["academic_motivation"]) * 2.5 +
    df["digital_addiction_score"] * 0.5 +
    df["brain_rot_index"] * 0.3
)

# Ensure no negative scores
df["academic_risk_score"] = df["academic_risk_score"].clip(lower=0)

print("academic_risk_score summary after engineering:")
print(df["academic_risk_score"].describe())

academic_risk_score summary after engineering:
count    100000.000000
mean         27.987109
std           8.827461
min           0.222360
25%          21.710423
50%          27.627668
75%          33.957008
max          61.479582
Name: academic_risk_score, dtype: float64


## 6. Round Float Columns to 2 Decimal Places

`student_id` is an integer column so it will not be affected by this step.

In [75]:
float_cols = df.select_dtypes(include="float").columns
df[float_cols] = df[float_cols].round(2)
print(f"Rounded {len(float_cols)} float column(s) to 2 decimal places.")

Rounded 33 float column(s) to 2 decimal places.


## 7. Final Check

A summary of the cleaned dataset: shape, dtypes, remaining nulls, and first 5 rows.

In [76]:
print("=" * 55)
print("FINAL CLEANED DATASET SUMMARY")
print("=" * 55)
print(f"Shape          : {df.shape}")
print(f"Total nulls    : {df.isnull().sum().sum()}")
print(f"Float columns  : {len(df.select_dtypes('float').columns)}")
print(f"Int columns    : {len(df.select_dtypes('int').columns)}")
print(f"Object columns : {len(df.select_dtypes('object').columns)}")
print(f"student_id range: {df['student_id'].min()} to {df['student_id'].max()}")
print("=" * 55)

FINAL CLEANED DATASET SUMMARY
Shape          : (100000, 48)
Total nulls    : 0
Float columns  : 33
Int columns    : 5
Object columns : 10
student_id range: 1 to 100000


In [77]:
print("Final dtypes:")
print(df.dtypes)

Final dtypes:
student_id                          int64
country                            object
development_level                  object
poverty_rate_percent              float64
internet_infrastructure_index     float64
average_internet_speed_mbps       float64
age                                 int64
gender                             object
urban_rural                        object
family_income_level                object
device_access                      object
internet_access_hours             float64
education_level                    object
field_of_study                     object
academic_motivation               float64
online_learning_hours             float64
social_media_hours                float64
sessions_per_day                  float64
average_session_length_minutes    float64
late_night_usage                   object
education_content_hours           float64
short_video_hours                 float64
entertainment_content_hours       float64
news_content_hours  

In [78]:
print("Null counts per column (final):")
print(df.isnull().sum())

Null counts per column (final):
student_id                        0
country                           0
development_level                 0
poverty_rate_percent              0
internet_infrastructure_index     0
average_internet_speed_mbps       0
age                               0
gender                            0
urban_rural                       0
family_income_level               0
device_access                     0
internet_access_hours             0
education_level                   0
field_of_study                    0
academic_motivation               0
online_learning_hours             0
social_media_hours                0
sessions_per_day                  0
average_session_length_minutes    0
late_night_usage                  0
education_content_hours           0
short_video_hours                 0
entertainment_content_hours       0
news_content_hours                0
likes_given_per_day               0
comments_written_per_day          0
posts_created_per_week          

In [79]:
# Verify the _per_day and _per_month columns are still numeric (not datetime)
per_day_cols = [c for c in df.columns if "_per_day" in c or "_per_week" in c or "_per_month" in c]
print("Dtypes of _per_day / _per_week / _per_month columns (must be numeric):")
print(df[per_day_cols].dtypes)
print("\nSample values:")
print(df[per_day_cols].head(3))

Dtypes of _per_day / _per_week / _per_month columns (must be numeric):
sessions_per_day              float64
likes_given_per_day           float64
comments_written_per_day      float64
posts_created_per_week        float64
study_hours_per_week          float64
ads_viewed_per_day            float64
ads_clicked_per_week          float64
digital_spending_per_month    float64
dtype: object

Sample values:
   sessions_per_day  likes_given_per_day  comments_written_per_day  \
0             13.97               132.14                     16.08   
1             11.72                94.87                      4.06   
2             10.86                99.80                      5.02   

   posts_created_per_week  study_hours_per_week  ads_viewed_per_day  \
0                    4.34                 26.25              139.74   
1                    7.08                 24.87              122.07   
2                    8.05                 19.88              133.58   

   ads_clicked_per_week  digi

In [80]:
print("First 5 rows of cleaned data:")
df.head()

First 5 rows of cleaned data:


,student_id,country,development_level,poverty_rate_percent,internet_infrastructure_index,average_internet_speed_mbps,age,gender,urban_rural,family_income_level,device_access,internet_access_hours,education_level,field_of_study,academic_motivation,online_learning_hours,social_media_hours,sessions_per_day,average_session_length_minutes,late_night_usage,education_content_hours,short_video_hours,entertainment_content_hours,news_content_hours,likes_given_per_day,comments_written_per_day,posts_created_per_week,late_night_score,brain_rot_index,brain_rot_level,attention_span_minutes,study_hours_per_week,class_attendance_rate,productivity_score,sleep_hours,stress_level,anxiety_score,depression_score,ads_viewed_per_day,ads_clicked_per_week,impulse_purchase_score,digital_spending_per_month,cyberbullying_exposure,adult_content_exposure,digital_addiction_score,wellbeing_index,academic_risk_score,financial_risk_score
0,1,Canada,Developed,14.51,90.50,188.19,15,Female,Urban,High,Both,6.25,School,Law,6.39,6.70,4.75,13.97,56.63,Sometimes,0.94,2.80,1.00,0.55,132.14,16.08,4.34,1,25.07,Medium,56.95,26.25,90.68,8.45,5.85,5.33,6.04,8.43,139.74,13.72,5.39,106.59,0,0,17.14,54.67,28.84,36.33
1,2,Vietnam,Developing,30.70,66.87,45.63,21,Female,Urban,Middle,Smartphone,6.90,Graduate,Social Sciences,8.00,7.36,4.27,11.72,53.36,Often,0.92,1.42,1.94,0.50,94.87,4.06,7.08,2,25.95,Medium,56.29,24.87,98.83,8.20,5.54,5.68,5.05,7.24,122.07,14.11,7.36,82.21,0,0,19.32,49.71,22.90,53.90
2,3,Netherlands,Developed,14.70,80.43,107.41,17,Female,Urban,Low,Smartphone,6.10,School,Law,3.66,5.45,4.43,10.86,52.52,Sometimes,0.35,1.44,2.65,0.62,99.80,5.02,8.05,1,29.85,Medium,47.74,19.88,84.45,5.76,5.97,5.30,4.16,6.21,133.58,18.37,6.72,26.59,0,0,15.68,51.47,38.87,51.88
3,4,Argentina,Developing,18.99,68.14,69.08,18,Female,Rural,Middle,Smartphone,5.98,Diploma,Social Sciences,6.35,7.62,4.32,12.39,46.99,Always,0.36,1.44,2.52,0.37,53.76,13.42,4.52,3,28.24,Medium,49.79,24.18,90.11,9.20,5.40,6.19,7.48,4.60,133.29,17.05,7.78,97.83,0,1,20.24,59.18,31.66,51.14
4,5,Vietnam,Developing,30.70,66.87,45.63,21,Female,Rural,Low,Shared Device,6.70,Graduate,Business,7.80,7.74,3.11,8.87,40.93,Never,0.65,1.23,1.23,0.39,81.35,7.92,3.82,0,10.99,Low,60.00,28.71,93.98,10.00,6.81,3.38,3.57,2.42,88.52,11.63,5.26,37.78,0,0,8.29,71.06,15.35,46.64


## 8. Store Cleaned DataFrame for Downstream Notebooks

In [81]:
# Assign to df_clean — this is the variable consumed by notebooks 03, 04, 05
df_clean = df.copy()

%store df_clean
print(f"df_clean stored. Shape: {df_clean.shape}")

Stored 'df_clean' (DataFrame)
df_clean stored. Shape: (100000, 48)


## Summary

- Loaded `df_sample` (100,000 rows) from notebook 01.
- Renamed all columns to snake_case.
- `student_id` retained (sequential 1..100,000 from notebook 01); only columns with >70% nulls are dropped.
- Replaced null sentinels with `NaN` and imputed: median for numeric, mode for categorical.
- `student_id` excluded from imputation and outlier treatment — it is an identifier, not a measure.
- Detected and capped outliers via IQR Winsorization (no rows deleted).
- **Fixed**: Date detection heuristic tightened — `_per_day`, `_per_week`, `_per_month` columns are
  no longer misidentified as datetime and incorrectly converted to epoch timestamps.
- **Added**: `brain_rot_level` rederived from `brain_rot_index` with thresholds:
  Low (<= 15), Medium (<= 30), High (> 30). This also resolves any null values in that column.
- **Added**: `academic_risk_score` engineered using a weighted formula based on class attendance, motivation, digital addiction, and brain rot.
- Encoded binary categoricals (Yes/No -> 1/0).
- Rounded all floats to 2 decimal places.
- `df_clean` stored for use in notebooks 03, 04, and 05.